# Provision VM — proj03

Provisions an `m1.xlarge` VM on **KVM@TACC**, attaches the persistent block volume, and configures all security groups.

**Prerequisites:**
- Run `provision_block.ipynb` first (block volume must exist)
- Run inside the Chameleon Jupyter environment
- SSH key uploaded to KVM@TACC

**Resources created:**
- Lease: `lease-<username>-proj03` (8 hours)
- Server: `node-<username>-proj03`
- Floating IP attached
- Block volume `block-<username>-proj03` attached at `/dev/vdb`
- Security groups: SSH, Postgres, RabbitMQ, MinIO, MLflow, PhotoPrism

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
project = "proj03"

print(f"User: {username} | Project: {project} | Site: KVM@TACC")

In [ ]:
# Reserve an m1.xlarge VM for 8 hours
l = lease.Lease(f"lease-{username}-{project}", duration=datetime.timedelta(hours=8))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)
print("Lease submitted. Waiting for ACTIVE state...")
l.wait()
print(f"Lease '{l.name}' is ACTIVE.")

In [ ]:
# Launch the VM instance
s = server.Server(
    f"node-{username}-{project}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

In [ ]:
s.associate_floating_ip()
s.refresh()
s.show(type="text")

In [ ]:
# Attach security groups for all data stack services
security_groups = [
    {"name": "allow-ssh",   "port": 22,   "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-5432",  "port": 5432, "description": "Enable TCP port 5432 (Postgres)"},
    {"name": "allow-5672",  "port": 5672, "description": "Enable TCP port 5672 (RabbitMQ AMQP)"},
    {"name": "allow-15672", "port": 15672,"description": "Enable TCP port 15672 (RabbitMQ Management UI)"},
    {"name": "allow-9000",  "port": 9000, "description": "Enable TCP port 9000 (MinIO API)"},
    {"name": "allow-9001",  "port": 9001, "description": "Enable TCP port 9001 (MinIO Console)"},
    {"name": "allow-5000",  "port": 5000, "description": "Enable TCP port 5000 (MLflow)"},
    {"name": "allow-30234", "port": 30234,"description": "Enable TCP port 30234 (PhotoPrism NodePort)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup(
        {
            "name": sg["name"],
            "description": sg["description"],
        }
    )
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")

In [ ]:
# Attach persistent block volume (must have been created by provision_block.ipynb)
cinder_client = chi.clients.cinder()
volume_name = f"block-{username}-{project}"
volumes = [v for v in cinder_client.volumes.list() if v.name == volume_name]

if not volumes:
    raise RuntimeError(
        f"Block volume '{volume_name}' not found. Run provision_block.ipynb first."
    )

volume = volumes[0]
s.attach_volume(volume.id)
print(f"Block volume '{volume.name}' attached at /dev/vdb.")

s.refresh()
floating_ip = next(
    addr["addr"]
    for addrs in s.addresses.values()
    for addr in addrs
    if addr.get("OS-EXT-IPS:type") == "floating"
)
print(f"\nVM ready. Floating IP: {floating_ip}")
print(f"SSH: ssh cc@{floating_ip}")
print("\n--- First time only: format and mount the block volume ---")
print("sudo parted -s /dev/vdb mklabel gpt && sudo parted -s /dev/vdb mkpart primary ext4 0% 100%")
print("sudo mkfs.ext4 /dev/vdb1")
print("sudo mkdir -p /mnt/block && sudo mount /dev/vdb1 /mnt/block")
print("sudo chown -R cc /mnt/block && sudo chgrp -R cc /mnt/block")
print("\n--- Subsequent sessions: just mount ---")
print("sudo mkdir -p /mnt/block && sudo mount /dev/vdb1 /mnt/block")